In [2]:
%pip install xgboost lightgbm

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
    --------------------------------------- 1.6/69.5 MB 9.2 MB/s eta 0:00:08
   - -------------------------------------- 3.4/69.5 MB 8.9 MB/s eta 0:00:08
   --- ------------------------------------ 5.2/69.5 MB 8.9 MB/s eta 0:00:08
   ---- ----------------------------------- 7.1/69.5 MB 8.8 MB/s eta 0:00:08
   ---- ----------------------------------- 8.4/69.5 MB 8.4 MB/s eta 0:00:08
   ----- ---------------------------------- 10.2/69.5 MB 8.3 MB/s eta 0:00:08
   ------ --------------------------------- 11.5/69.5 MB 8.2 MB/s eta 0:00:08
   ------- -------------------------------- 13.4/69.5 MB 8.3 MB/s eta 0:00:07
   -------- ------------------------------- 15.2/69.5 MB 8.3 MB/s eta 0:00:07
   --------- ------------------------------ 17.0/69.5 MB 8.4 MB/s eta 0:00:07
   ---------- ----------------------------- 18.9/69.5 MB 8.4 MB/s eta 0:00:07
   ----------- ---------------------------- 20.7/69.5 MB 8.5 MB/s eta 0:00:06

# Notebook 07 — Baseline Models
## Section 1: Naive + Linear Family Baselines (MAE)

First regression models for **`delay_in_min`**.

**Models**
1. Naive mean predictor  
2. Naive median predictor  
3. Linear Regression  
4. Ridge Regression  
5. Lasso Regression  
6. Elastic Net  

Each linear model is trained on:
- operational features only (RQ1)  
- operational + weather (RQ2 check)

**Metric:** MAE only (minutes)  
**Data:** `ice_train.parquet` → train | `ice_test.parquet` → test (Sep)

**Answers:** RQ1 (operational only) and first check for RQ2 (weather added)

In [1]:
# =============================================================================
# Notebook 07 | Section 1: Baseline Regression Models (MAE)
# =============================================================================
# Load train/test from disk → naive + Linear/Ridge/Lasso/ElasticNet.
# Evaluate with MAE only. Saves baseline_results.json
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
RESULTS_PATH = REFERENCE_DIR / "baseline_results.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )


def evaluate_model(name: str, y_train, y_test, y_train_pred, y_test_pred, feature_set: str) -> dict:
    train_mae = round(float(mean_absolute_error(y_train, y_train_pred)), 4)
    test_mae = round(float(mean_absolute_error(y_test, y_test_pred)), 4)
    return {
        "model": name,
        "feature_set": feature_set,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "metric": "mae",
    }


def train_sklearn_model(name, model, X_train, X_test, y_train, y_test, feature_set: str) -> dict:
    model.fit(X_train, y_train)
    return evaluate_model(
        name=name,
        y_train=y_train,
        y_test=y_test,
        y_train_pred=model.predict(X_train),
        y_test_pred=model.predict(X_test),
        feature_set=feature_set,
    )


def run_linear_family(feature_cols, feature_set_name, train_df, test_df, y_train, y_test):
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    models = [
        ("LinearRegression", LinearRegression()),
        ("Ridge", Ridge(alpha=1.0, random_state=42)),
        ("Lasso", Lasso(alpha=0.01, max_iter=5000, random_state=42)),
        ("ElasticNet", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=42)),
    ]

    out = []
    for name, est in models:
        pipe = Pipeline([("prep", build_preprocessor(feature_cols)), ("model", est)])
        res = train_sklearn_model(name, pipe, X_train, X_test, y_train, y_test, feature_set_name)
        out.append(res)
        print(f"  {name:<16} → test MAE = {res['test_mae']:.4f} min")
    return out


# =============================================================================
# LOAD DATA + CONFIG FROM DISK
# =============================================================================
split_cfg = load_json(SPLIT_CONFIG_PATH)
operational_features = split_cfg["feature_sets_for_ablation"]["operational_only"]
full_features = split_cfg["feature_sets_for_ablation"]["full_with_weather"]

print("Notebook 07 | Section 1 — Baseline models (MAE)")
print(f"Target : {PRIMARY_TARGET}")
print(f"Train  : {split_cfg['split']['train_months']}")
print(f"Test   : {split_cfg['split']['test_months']}")
print()

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Missing train/test parquets. Run Notebook 06 Section 2.")

print(f"Loading train: {TRAIN_PATH.name}")
train_df = pd.read_parquet(TRAIN_PATH)
print(f"Loading test : {TEST_PATH.name}")
test_df = pd.read_parquet(TEST_PATH)

y_train = train_df[PRIMARY_TARGET].values
y_test = test_df[PRIMARY_TARGET].values

print(f"Train rows: {len(train_df):,}")
print(f"Test rows : {len(test_df):,}")
print()

results: list[dict] = []

# =============================================================================
# 1) NAIVE BASELINES
# =============================================================================
print("=" * 72)
print("NAIVE BASELINES")
print("=" * 72)

mean_pred = float(np.mean(y_train))
median_pred = float(np.median(y_train))

naive_mean = evaluate_model(
    "NaiveMean",
    y_train,
    y_test,
    np.full_like(y_train, mean_pred, dtype=float),
    np.full_like(y_test, mean_pred, dtype=float),
    feature_set="none",
)
naive_median = evaluate_model(
    "NaiveMedian",
    y_train,
    y_test,
    np.full_like(y_train, median_pred, dtype=float),
    np.full_like(y_test, median_pred, dtype=float),
    feature_set="none",
)

results.extend([naive_mean, naive_median])

print(f"  Naive mean   (predict {mean_pred:.2f}) → test MAE = {naive_mean['test_mae']:.4f} min")
print(f"  Naive median (predict {median_pred:.2f}) → test MAE = {naive_median['test_mae']:.4f} min")
print()

# =============================================================================
# 2) LINEAR FAMILY — OPERATIONAL ONLY (RQ1)
# =============================================================================
print("=" * 72)
print("OPERATIONAL FEATURES ONLY (RQ1)")
print("=" * 72)

res_op = run_linear_family(
    operational_features, "operational_only", train_df, test_df, y_train, y_test
)
results.extend(res_op)
print()

# =============================================================================
# 3) LINEAR FAMILY — FULL + WEATHER (RQ2)
# =============================================================================
print("=" * 72)
print("OPERATIONAL + WEATHER (RQ2 check)")
print("=" * 72)

res_full = run_linear_family(
    full_features, "full_with_weather", train_df, test_df, y_train, y_test
)
results.extend(res_full)
print()


def _mae(model_name: str, feature_set: str) -> float:
    return next(
        r["test_mae"]
        for r in results
        if r["model"] == model_name and r["feature_set"] == feature_set
    )


res_ridge_op_mae = _mae("Ridge", "operational_only")
res_ridge_full_mae = _mae("Ridge", "full_with_weather")

# =============================================================================
# 4) SUMMARY TABLE + SAVE
# =============================================================================
results_df = pd.DataFrame(results).sort_values("test_mae")
best = results_df.iloc[0]

print("=" * 72)
print("RESULTS (sorted by test MAE — lower is better)")
print("=" * 72)
print(results_df.to_string(index=False))
print()
print(f"Best baseline: {best['model']} ({best['feature_set']}) → test MAE = {best['test_mae']:.4f} min")
print(f"Naive median benchmark: {naive_median['test_mae']:.4f} min")
print()

beats_naive = results_df[results_df["test_mae"] < naive_median["test_mae"]]
print(f"Models beating naive median: {len(beats_naive)} / {len(results_df)}")

weather_gain = res_ridge_op_mae - res_ridge_full_mae
print(
    f"Ridge weather gain (operational - full): {weather_gain:.4f} min "
    f"({'weather helps' if weather_gain > 0 else 'no gain'})"
)
print()

report = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "train_parquet": str(TRAIN_PATH),
        "test_parquet": str(TEST_PATH),
        "models": [
            "NaiveMean",
            "NaiveMedian",
            "LinearRegression",
            "Ridge",
            "Lasso",
            "ElasticNet",
        ],
    },
    "naive_reference": {
        "mean_predictor": mean_pred,
        "median_predictor": median_pred,
        "naive_mean_test_mae": naive_mean["test_mae"],
        "naive_median_test_mae": naive_median["test_mae"],
    },
    "feature_sets": {
        "operational_only": operational_features,
        "full_with_weather": full_features,
    },
    "results": results,
    "best_model": {
        "model": best["model"],
        "feature_set": best["feature_set"],
        "test_mae": float(best["test_mae"]),
    },
    "rq_notes": {
        "RQ1_operational_best_test_mae": min(r["test_mae"] for r in res_op),
        "RQ2_weather_ridge_gain_vs_operational": round(float(weather_gain), 4),
    },
}

with RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE")
print(sep)
print(f"Saved: {RESULTS_PATH}")
print("Next: Section 2 — Random Forest")
print(sep)

Notebook 07 | Section 1 — Baseline models (MAE)
Target : delay_in_min
Train  : ['2024-07', '2024-08']
Test   : ['2024-09']

Loading train: ice_train.parquet
Loading test : ice_test.parquet
Train rows: 255,062
Test rows : 121,964

NAIVE BASELINES
  Naive mean   (predict 11.04) → test MAE = 11.5078 min
  Naive median (predict 4.00) → test MAE = 9.4813 min

OPERATIONAL FEATURES ONLY (RQ1)
  LinearRegression → test MAE = 11.2068 min
  Ridge            → test MAE = 11.2051 min
  Lasso            → test MAE = 11.5309 min
  ElasticNet       → test MAE = 11.5822 min

OPERATIONAL + WEATHER (RQ2 check)
  LinearRegression → test MAE = 10.5935 min
  Ridge            → test MAE = 10.5879 min
  Lasso            → test MAE = 10.7158 min
  ElasticNet       → test MAE = 10.7547 min

RESULTS (sorted by test MAE — lower is better)
           model       feature_set  train_mae  test_mae metric
     NaiveMedian              none     9.9773    9.4813    mae
           Ridge full_with_weather    10.1858   10

# Notebook 07 — Baseline Models
## Section 2: Random Forest (Bagging Ensemble, MAE)

Non-linear bagging model to beat linear baselines and naive median.

**Runs**
1. Random Forest — operational features only (RQ1)  
2. Random Forest — operational + weather (RQ2)

**Metric:** MAE (minutes)  
**Compare to:** Naive median (~9.48 min) and linear family from Section 1

**Next after this:** Section 3 — Gradient Boosting / XGBoost / LightGBM

In [2]:
# =============================================================================
# Notebook 07 | Section 2: Random Forest (MAE)
# =============================================================================
# Load train/test from disk → RF on operational vs full+weather.
# Saves random_forest_results.json
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
BASELINE_RESULTS_PATH = REFERENCE_DIR / "baseline_results.json"
RF_RESULTS_PATH = REFERENCE_DIR / "random_forest_results.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 20,
    "min_samples_leaf": 10,
    "random_state": 42,
    "n_jobs": -1,
}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )


def get_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    try:
        return list(preprocessor.get_feature_names_out())
    except Exception:
        return []


def train_rf(name: str, feature_cols: list[str], train_df, test_df, feature_set: str) -> dict:
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df[PRIMARY_TARGET].values
    y_test = test_df[PRIMARY_TARGET].values

    preprocessor = build_preprocessor(feature_cols)
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", RandomForestRegressor(**RF_PARAMS)),
        ]
    )

    print(f"Training {name} ({feature_set}) ...")
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae = round(float(mean_absolute_error(y_train, y_train_pred)), 4)
    test_mae = round(float(mean_absolute_error(y_test, y_test_pred)), 4)

    rf = model.named_steps["model"]
    prep = model.named_steps["prep"]
    feat_names = get_feature_names(prep)
    importances = rf.feature_importances_

    if len(feat_names) == len(importances):
        imp_df = (
            pd.DataFrame({"feature": feat_names, "importance": importances})
            .sort_values("importance", ascending=False)
            .head(15)
        )
        top_features = imp_df.to_dict(orient="records")
    else:
        top_features = []

    print(f"  → train MAE = {train_mae:.4f} | test MAE = {test_mae:.4f}")
    print()

    return {
        "model": name,
        "feature_set": feature_set,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "metric": "mae",
        "rf_params": RF_PARAMS,
        "top_feature_importances": top_features,
    }


split_cfg = load_json(SPLIT_CONFIG_PATH)
baseline = load_json(BASELINE_RESULTS_PATH) if BASELINE_RESULTS_PATH.exists() else {}

operational_features = split_cfg["feature_sets_for_ablation"]["operational_only"]
full_features = split_cfg["feature_sets_for_ablation"]["full_with_weather"]

print("Notebook 07 | Section 2 — Random Forest (MAE)")
print(f"Target : {PRIMARY_TARGET}")
print(f"Train  : {split_cfg['split']['train_months']}")
print(f"Test   : {split_cfg['split']['test_months']}")
print(f"RF params: {RF_PARAMS}")
print("(Training may take a few minutes...)")
print()

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
print(f"Train rows: {len(train_df):,}")
print(f"Test rows : {len(test_df):,}")
print()

results = []

print("=" * 72)
print("RANDOM FOREST")
print("=" * 72)

results.append(
    train_rf("RandomForest", operational_features, train_df, test_df, "operational_only")
)
results.append(
    train_rf("RandomForest", full_features, train_df, test_df, "full_with_weather")
)

naive_median_mae = baseline.get("naive_reference", {}).get("naive_median_test_mae")
best_ridge_mae = None
if baseline.get("results"):
    ridge_full = [
        r for r in baseline["results"]
        if r["model"] == "Ridge" and r["feature_set"] == "full_with_weather"
    ]
    if ridge_full:
        best_ridge_mae = ridge_full[0]["test_mae"]

results_df = pd.DataFrame(results).sort_values("test_mae")
best_rf = results_df.iloc[0]

op_mae = results_df.loc[results_df["feature_set"] == "operational_only", "test_mae"].iloc[0]
full_mae = results_df.loc[results_df["feature_set"] == "full_with_weather", "test_mae"].iloc[0]
weather_gain = round(float(op_mae - full_mae), 4)

print("=" * 72)
print("RESULTS")
print("=" * 72)
print(results_df[["model", "feature_set", "train_mae", "test_mae"]].to_string(index=False))
print()
print(f"Best RF test MAE     : {best_rf['test_mae']:.4f} min")
if naive_median_mae is not None:
    print(f"Naive median MAE     : {naive_median_mae:.4f} min")
    print(f"Beats naive median?  : {'YES' if best_rf['test_mae'] < naive_median_mae else 'NO'}")
if best_ridge_mae is not None:
    print(f"Best Ridge MAE (S1)  : {best_ridge_mae:.4f} min")
print(f"Weather gain (RF)    : {weather_gain:.4f} min")
print()

report = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "model_family": "RandomForestRegressor",
        "note": "Bagging ensemble. Boosting models are in Section 3.",
    },
    "rf_params": RF_PARAMS,
    "results": results,
    "best_rf": {
        "feature_set": best_rf["feature_set"],
        "test_mae": float(best_rf["test_mae"]),
    },
    "comparison": {
        "naive_median_test_mae": naive_median_mae,
        "best_ridge_test_mae_section1": best_ridge_mae,
        "weather_gain_operational_minus_full": weather_gain,
        "beats_naive_median": bool(best_rf["test_mae"] < naive_median_mae)
        if naive_median_mae is not None
        else None,
    },
    "rq_notes": {
        "RQ1_rf_operational_test_mae": float(op_mae),
        "RQ2_rf_full_test_mae": float(full_mae),
        "RQ2_weather_improves_mae": bool(weather_gain > 0),
    },
}

with RF_RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE")
print(sep)
print(f"Saved: {RF_RESULTS_PATH}")
print("Next: Section 3 — Boosting models (GB / XGBoost / LightGBM)")
print(sep)

Notebook 07 | Section 2 — Random Forest (MAE)
Target : delay_in_min
Train  : ['2024-07', '2024-08']
Test   : ['2024-09']
RF params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 10, 'random_state': 42, 'n_jobs': -1}
(Training may take a few minutes...)

Train rows: 255,062
Test rows : 121,964

RANDOM FOREST
Training RandomForest (operational_only) ...
  → train MAE = 9.6226 | test MAE = 10.4758

Training RandomForest (full_with_weather) ...
  → train MAE = 9.2731 | test MAE = 10.4980

RESULTS
       model       feature_set  train_mae  test_mae
RandomForest  operational_only     9.6226   10.4758
RandomForest full_with_weather     9.2731   10.4980

Best RF test MAE     : 10.4758 min
Naive median MAE     : 9.4813 min
Beats naive median?  : NO
Best Ridge MAE (S1)  : 10.5879 min
Weather gain (RF)    : -0.0222 min

Section 2 COMPLETE
Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\random_forest_results.json
Next: Section 3 — Boosting models (GB / XGBoost / 

# Notebook 07 — Baseline Models
## Section 3: Boosting Family (MAE)

Same boosting idea, three implementations — fair within-family comparison.

**Models**
1. Gradient Boosting (sklearn)  
2. XGBoost  
3. LightGBM  

**Runs each on**
- operational features only (RQ1)  
- operational + weather (RQ2)

**Metric:** MAE (minutes)  
**Compare to:** Naive median, linear family (S1), Random Forest (S2)

**Note:** Uses a train sample (80,000 rows) for speed — same sample for all three boosters.

In [3]:
# =============================================================================
# Notebook 07 | Section 3: Boosting Family (MAE)
# =============================================================================
# GradientBoosting + XGBoost + LightGBM on operational vs full+weather.
# Saves boosting_results.json
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from xgboost import XGBRegressor
except ImportError as e:
    raise ImportError("Install xgboost: pip install xgboost") from e

try:
    from lightgbm import LGBMRegressor
except ImportError as e:
    raise ImportError("Install lightgbm: pip install lightgbm") from e

USE_TRAIN_SAMPLE = True
TRAIN_SAMPLE_SIZE = 80_000
RANDOM_STATE = 42

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]

GB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "random_state": RANDOM_STATE,
}

XGB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist",
}

LGBM_PARAMS = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
BASELINE_RESULTS_PATH = REFERENCE_DIR / "baseline_results.json"
RF_RESULTS_PATH = REFERENCE_DIR / "random_forest_results.json"
BOOSTING_RESULTS_PATH = REFERENCE_DIR / "boosting_results.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )


def train_booster(name: str, estimator, feature_cols, fit_df, test_df, feature_set: str) -> dict:
    X_train = fit_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = fit_df[PRIMARY_TARGET].values
    y_test = test_df[PRIMARY_TARGET].values

    pipe = Pipeline(
        steps=[
            ("prep", build_preprocessor(feature_cols)),
            ("model", estimator),
        ]
    )

    print(f"Training {name} ({feature_set}) on {len(fit_df):,} rows ...")
    pipe.fit(X_train, y_train)

    train_mae = round(float(mean_absolute_error(y_train, pipe.predict(X_train))), 4)
    test_mae = round(float(mean_absolute_error(y_test, pipe.predict(X_test))), 4)
    print(f"  → train MAE = {train_mae:.4f} | test MAE = {test_mae:.4f}")
    print()

    return {
        "model": name,
        "feature_set": feature_set,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "metric": "mae",
        "train_rows_used": int(len(fit_df)),
    }


split_cfg = load_json(SPLIT_CONFIG_PATH)
baseline = load_json(BASELINE_RESULTS_PATH) if BASELINE_RESULTS_PATH.exists() else {}
rf_report = load_json(RF_RESULTS_PATH) if RF_RESULTS_PATH.exists() else {}

operational_features = split_cfg["feature_sets_for_ablation"]["operational_only"]
full_features = split_cfg["feature_sets_for_ablation"]["full_with_weather"]

print("Notebook 07 | Section 3 — Boosting family (MAE)")
print(f"Target : {PRIMARY_TARGET}")
print(f"Train  : {split_cfg['split']['train_months']}")
print(f"Test   : {split_cfg['split']['test_months']}")
print()

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

if USE_TRAIN_SAMPLE and len(train_df) > TRAIN_SAMPLE_SIZE:
    fit_df = train_df.sample(n=TRAIN_SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"Using train sample: {len(fit_df):,} / {len(train_df):,}")
else:
    fit_df = train_df
    print(f"Using full train: {len(fit_df):,}")

print(f"Test rows : {len(test_df):,}")
print()

results = []
print("=" * 72)
print("BOOSTING MODELS")
print("=" * 72)

for name, params in [
    ("GradientBoosting", GB_PARAMS),
    ("XGBoost", XGB_PARAMS),
    ("LightGBM", LGBM_PARAMS),
]:
    for feature_cols, feature_set in [
        (operational_features, "operational_only"),
        (full_features, "full_with_weather"),
    ]:
        if name == "GradientBoosting":
            estimator = GradientBoostingRegressor(**params)
        elif name == "XGBoost":
            estimator = XGBRegressor(**params)
        else:
            estimator = LGBMRegressor(**params)

        results.append(
            train_booster(name, estimator, feature_cols, fit_df, test_df, feature_set)
        )

results_df = pd.DataFrame(results).sort_values("test_mae")
best = results_df.iloc[0]
naive_median_mae = baseline.get("naive_reference", {}).get("naive_median_test_mae")
best_rf_mae = rf_report.get("best_rf", {}).get("test_mae")

print("=" * 72)
print("RESULTS")
print("=" * 72)
print(results_df[["model", "feature_set", "train_mae", "test_mae"]].to_string(index=False))
print()
print(f"Best boosting MAE : {best['model']} ({best['feature_set']}) = {best['test_mae']:.4f} min")
if best_rf_mae is not None:
    print(f"Best RF MAE (S2)  : {best_rf_mae:.4f} min")
    print(f"Beats RF?         : {'YES' if best['test_mae'] < best_rf_mae else 'NO'}")
if naive_median_mae is not None:
    print(f"Naive median MAE  : {naive_median_mae:.4f} min")
    print(f"Beats naive?      : {'YES' if best['test_mae'] < naive_median_mae else 'NO'}")
print()

report = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "model_family": "boosting",
        "use_train_sample": USE_TRAIN_SAMPLE,
        "train_sample_size": TRAIN_SAMPLE_SIZE if USE_TRAIN_SAMPLE else None,
        "train_rows_used": int(len(fit_df)),
    },
    "params": {
        "GradientBoosting": GB_PARAMS,
        "XGBoost": XGB_PARAMS,
        "LightGBM": LGBM_PARAMS,
    },
    "results": results,
    "best_boosting": {
        "model": best["model"],
        "feature_set": best["feature_set"],
        "test_mae": float(best["test_mae"]),
    },
    "comparison": {
        "naive_median_test_mae": naive_median_mae,
        "best_rf_test_mae_section2": best_rf_mae,
        "beats_rf": bool(best["test_mae"] < best_rf_mae) if best_rf_mae is not None else None,
        "beats_naive_median": bool(best["test_mae"] < naive_median_mae)
        if naive_median_mae is not None
        else None,
    },
}

with BOOSTING_RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 3 COMPLETE")
print(sep)
print(f"Saved: {BOOSTING_RESULTS_PATH}")
print("Next: Section 4 — Notebook 07 close-out")
print(sep)

Notebook 07 | Section 3 — Boosting family (MAE)
Target : delay_in_min
Train  : ['2024-07', '2024-08']
Test   : ['2024-09']

Using train sample: 80,000 / 255,062
Test rows : 121,964

BOOSTING MODELS
Training GradientBoosting (operational_only) on 80,000 rows ...
  → train MAE = 10.0904 | test MAE = 10.5635

Training GradientBoosting (full_with_weather) on 80,000 rows ...
  → train MAE = 10.0181 | test MAE = 10.3750

Training XGBoost (operational_only) on 80,000 rows ...
  → train MAE = 9.9585 | test MAE = 10.5492

Training XGBoost (full_with_weather) on 80,000 rows ...
  → train MAE = 9.8496 | test MAE = 10.3111

Training LightGBM (operational_only) on 80,000 rows ...


c:\Users\Manikanta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Manikanta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  → train MAE = 10.1006 | test MAE = 10.5526

Training LightGBM (full_with_weather) on 80,000 rows ...


c:\Users\Manikanta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Manikanta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  → train MAE = 10.0163 | test MAE = 10.3389

RESULTS
           model       feature_set  train_mae  test_mae
         XGBoost full_with_weather     9.8496   10.3111
        LightGBM full_with_weather    10.0163   10.3389
GradientBoosting full_with_weather    10.0181   10.3750
         XGBoost  operational_only     9.9585   10.5492
        LightGBM  operational_only    10.1006   10.5526
GradientBoosting  operational_only    10.0904   10.5635

Best boosting MAE : XGBoost (full_with_weather) = 10.3111 min
Best RF MAE (S2)  : 10.4758 min
Beats RF?         : YES
Naive median MAE  : 9.4813 min
Beats naive?      : NO

Section 3 COMPLETE
Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\boosting_results.json
Next: Section 4 — Notebook 07 close-out


# Notebook 07 — Baseline Models
## Section 4: Close-Out

Verify baseline + Random Forest + boosting results on disk and summarize full MAE leaderboard.

**Next:** Notebook 08 — Hyperparameter tuning + weather ablation + final model selection

In [4]:
# =============================================================================
# Notebook 07 | Section 4: Close-Out
# =============================================================================
# Verify model result files; summarize MAE comparison; save notebook_07_summary.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUMMARY_PATH = REFERENCE_DIR / "notebook_07_summary.json"

BASELINE_PATH = REFERENCE_DIR / "baseline_results.json"
RF_PATH = REFERENCE_DIR / "random_forest_results.json"
BOOSTING_PATH = REFERENCE_DIR / "boosting_results.json"
SPLIT_PATH = REFERENCE_DIR / "train_test_split_config.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


print("Notebook 07 | Section 4 — Close-out")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    size = path.stat().st_size if ok else 0
    checklist.append(
        {
            "label": label,
            "path": str(path),
            "exists": ok,
            "size_mb": round(size / 1e6, 4),
        }
    )
    print(f"  [{'OK' if ok else 'MISSING'}] {label}")
    return ok


print("=" * 72)
print("FILE CHECKLIST")
print("=" * 72)

all_ok = True
all_ok &= check("ice_train.parquet", PROCESSED_DIR / "ice_train.parquet")
all_ok &= check("ice_test.parquet", PROCESSED_DIR / "ice_test.parquet")
all_ok &= check("train_test_split_config.json", SPLIT_PATH)
all_ok &= check("baseline_results.json", BASELINE_PATH)
all_ok &= check("random_forest_results.json", RF_PATH)
all_ok &= check("boosting_results.json", BOOSTING_PATH)

baseline = load_json(BASELINE_PATH)
rf = load_json(RF_PATH)
boosting = load_json(BOOSTING_PATH)
split_cfg = load_json(SPLIT_PATH)

leaderboard = []

for r in baseline["results"]:
    leaderboard.append(
        {
            "model": r["model"],
            "feature_set": r["feature_set"],
            "test_mae": r["test_mae"],
            "section": "Section 1",
        }
    )

for r in rf["results"]:
    leaderboard.append(
        {
            "model": r["model"],
            "feature_set": r["feature_set"],
            "test_mae": r["test_mae"],
            "section": "Section 2",
        }
    )

for r in boosting["results"]:
    leaderboard.append(
        {
            "model": r["model"],
            "feature_set": r["feature_set"],
            "test_mae": r["test_mae"],
            "section": "Section 3",
        }
    )

lb_df = pd.DataFrame(leaderboard).sort_values("test_mae")
best_overall = lb_df.iloc[0]
best_ml = lb_df[~lb_df["model"].isin(["NaiveMean", "NaiveMedian"])].iloc[0]

naive_median = baseline["naive_reference"]["naive_median_test_mae"]
rf_op = rf["rq_notes"]["RQ1_rf_operational_test_mae"]
rf_full = rf["rq_notes"]["RQ2_rf_full_test_mae"]
best_boost = boosting["best_boosting"]

print()
print("=" * 72)
print("LEADERBOARD (test MAE — lower is better)")
print("=" * 72)
print(lb_df.to_string(index=False))
print()
print(
    f"Best overall        : {best_overall['model']} ({best_overall['feature_set']}) = {best_overall['test_mae']:.4f} min"
)
print(
    f"Best ML model       : {best_ml['model']} ({best_ml['feature_set']}) = {best_ml['test_mae']:.4f} min"
)
print(f"Naive median        : {naive_median:.4f} min")
print(f"Best ML beats naive?: {'YES' if best_ml['test_mae'] < naive_median else 'NO'}")
print(
    f"Best boosting       : {best_boost['model']} ({best_boost['feature_set']}) = {best_boost['test_mae']:.4f} min"
)
print(f"Weather helps RF?   : {'YES' if rf_full < rf_op else 'NO (operational slightly better)'}")
print()

print("=" * 72)
print("KEY FINDINGS")
print("=" * 72)
print("  1. Metric = MAE (minutes) on Sep test set")
print("  2. Linear family includes Ridge + Lasso + ElasticNet")
print("  3. Random Forest = bagging baseline for tree models")
print("  4. Boosting compared: GradientBoosting / XGBoost / LightGBM")
print("  5. Best ML so far from leaderboard above (likely XGBoost + weather)")
print()

ready = bool(all_ok)
print("=" * 72)
print(f"Notebook 07 complete: {'YES' if ready else 'NO'}")
print("=" * 72)
if not ready:
    missing = [c["label"] for c in checklist if not c["exists"]]
    raise FileNotFoundError(f"Missing: {missing}")

summary = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 4",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": "delay_in_min",
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "checklist": checklist,
    "all_files_ok": ready,
    "split": split_cfg["split"],
    "leaderboard": leaderboard,
    "best_overall": {
        "model": best_overall["model"],
        "feature_set": best_overall["feature_set"],
        "test_mae": float(best_overall["test_mae"]),
    },
    "best_ml_model": {
        "model": best_ml["model"],
        "feature_set": best_ml["feature_set"],
        "test_mae": float(best_ml["test_mae"]),
    },
    "key_findings": {
        "naive_median_test_mae": naive_median,
        "best_rf_test_mae": float(min(rf_op, rf_full)),
        "best_boosting_test_mae": float(best_boost["test_mae"]),
        "best_boosting_model": best_boost["model"],
        "best_ml_beats_naive_median": bool(best_ml["test_mae"] < naive_median),
        "weather_helps_rf": bool(rf_full < rf_op),
        "summary_sentence": (
            f"Best ML: {best_ml['model']} ({best_ml['feature_set']}) "
            f"with test MAE {best_ml['test_mae']:.4f} min vs naive median {naive_median:.4f} min."
        ),
    },
    "next_notebook": {
        "id": "08",
        "goal": "RF tuning + full leaderboard + weather ablation + final model selection",
    },
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print()
print(f"Saved: {SUMMARY_PATH}")
print("Notebook 07 DONE.")

Notebook 07 | Section 4 — Close-out

FILE CHECKLIST
  [OK] ice_train.parquet
  [OK] ice_test.parquet
  [OK] train_test_split_config.json
  [OK] baseline_results.json
  [OK] random_forest_results.json
  [OK] boosting_results.json

LEADERBOARD (test MAE — lower is better)
           model       feature_set  test_mae   section
     NaiveMedian              none    9.4813 Section 1
         XGBoost full_with_weather   10.3111 Section 3
        LightGBM full_with_weather   10.3389 Section 3
GradientBoosting full_with_weather   10.3750 Section 3
    RandomForest  operational_only   10.4758 Section 2
    RandomForest full_with_weather   10.4980 Section 2
         XGBoost  operational_only   10.5492 Section 3
        LightGBM  operational_only   10.5526 Section 3
GradientBoosting  operational_only   10.5635 Section 3
           Ridge full_with_weather   10.5879 Section 1
LinearRegression full_with_weather   10.5935 Section 1
           Lasso full_with_weather   10.7158 Section 1
      ElasticN